# So sánh mô hình đơn lẻ và mô hình tổ hợp

*Bài 9, mục 9.2.2.* So sánh trên dữ liệu ung thư vú bằng **kiểm định chéo lặp lại** (5 phần × 5 lần lặp): độ chính xác trung bình, độ lệch chuẩn (mức ổn định) và thời gian huấn luyện.

In [1]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import (AdaBoostClassifier, BaggingClassifier, GradientBoostingClassifier,
                              RandomForestClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
models = {
    'Cây quyết định': DecisionTreeClassifier(random_state=0),
    'Hồi quy Logistic': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'SVM RBF': make_pipeline(StandardScaler(), SVC()),
    'Bagging (100 cây)': BaggingClassifier(DecisionTreeClassifier(), n_estimators=100, random_state=0, n_jobs=-1),
    'Random Forest (100 cây)': RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1),
    'AdaBoost (200 cây 1 tầng)': AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=200, random_state=0),
    'Gradient Boosting (200 cây)': GradientBoostingClassifier(n_estimators=200, random_state=0),
    'Stacking (Logistic + SVM + cây)': StackingClassifier(
        [('logistic', make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
         ('svm', make_pipeline(StandardScaler(), SVC())),
         ('tree', DecisionTreeClassifier(max_depth=4, random_state=0))],
        final_estimator=LogisticRegression(), cv=5),
}
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=0)

In [2]:
rows = []
for name, model in models.items():
    result = cross_validate(model, X, y, cv=cv, return_train_score=True)
    rows.append([name, result['train_score'].mean(), result['test_score'].mean(),
                 result['test_score'].std(), result['fit_time'].mean()])
table = pd.DataFrame(rows, columns=['mô hình', 'huấn luyện', 'kiểm tra (TB)', 'kiểm tra (độ lệch chuẩn)', 'thời gian huấn luyện (s)'])
table.round(4)

,mô hình,huấn luyện,kiểm tra (TB),kiểm tra (độ lệch chuẩn),thời gian huấn luyện (s)
0,Cây quyết định,1.0000,0.9283,0.0196,0.0069
1,Hồi quy Logistic,0.9885,0.9772,0.0153,0.0070
2,SVM RBF,0.9873,0.9736,0.0131,0.0045
3,Bagging (100 cây),1.0000,0.9585,0.0178,0.4520
4,Random Forest (100 cây),1.0000,0.9599,0.0179,0.1636
5,AdaBoost (200 cây 1 tầng),1.0000,0.9708,0.0144,0.5370
6,Gradient Boosting (200 cây),1.0000,0.9634,0.0212,0.6247
7,Stacking (Logistic + SVM + cây),0.9893,0.9754,0.0153,0.1199


- Cây quyết định đơn lẻ: độ chính xác huấn luyện 1.0 nhưng kiểm tra thấp nhất và dao động nhiều nhất giữa các lần chia (phương sai cao).
- Bagging, Random Forest giảm phương sai: độ chính xác tăng rõ so với một cây, độ lệch chuẩn giảm.
- Boosting giảm cả thiên lệch: từ các cây một tầng rất yếu tạo thành mô hình mạnh.
- Trên dữ liệu này, hồi quy Logistic và SVM (đã chuẩn hóa) vốn đã rất tốt vì hai lớp gần như tách được tuyến tính; các mô hình tổ hợp cây không vượt qua được chúng. Học tổ hợp cải thiện mạnh nhất cho các mô hình có phương sai cao như cây quyết định.
- Đổi lại, mô hình tổ hợp tốn thời gian huấn luyện hơn và khó giải thích hơn.

## 2. Độ chính xác theo số mô hình thành phần

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
sizes = [1, 2, 5, 10, 20, 50, 100, 200]
for name, make_model in [('Random Forest', lambda n: RandomForestClassifier(n_estimators=n, random_state=0, n_jobs=-1)),
                         ('AdaBoost', lambda n: AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=n, random_state=0)),
                         ('Gradient Boosting', lambda n: GradientBoostingClassifier(n_estimators=n, random_state=0))]:
    scores = [make_model(n).fit(X_train, y_train).score(X_test, y_test) for n in sizes]
    print('%-17s' % name, ' '.join('%.3f' % s for s in scores))
print('Số mô hình       ', ' '.join('%5d' % n for n in sizes))

Random Forest     0.889 0.883 0.942 0.936 0.942 0.947 0.953 0.953


AdaBoost          0.889 0.889 0.930 0.942 0.936 0.942 0.942 0.959


Gradient Boosting 0.626 0.626 0.906 0.930 0.936 0.936 0.936 0.942
Số mô hình            1     2     5    10    20    50   100   200


## 3. Dữ liệu phi tuyến: hai nửa vầng trăng

Khi ranh giới giữa hai lớp phi tuyến, mô hình tuyến tính không còn lợi thế.

In [4]:
from sklearn.datasets import make_moons

X_moons, y_moons = make_moons(n_samples=500, noise=0.3, random_state=0)
moons_models = {
    'Cây quyết định': DecisionTreeClassifier(random_state=0),
    'Hồi quy Logistic': LogisticRegression(),
    'Random Forest (100 cây)': RandomForestClassifier(n_estimators=100, random_state=0, n_jobs=-1),
    'AdaBoost (200 cây 1 tầng)': AdaBoostClassifier(DecisionTreeClassifier(max_depth=1), n_estimators=200, random_state=0),
    'Gradient Boosting (200 cây)': GradientBoostingClassifier(n_estimators=200, max_depth=2, random_state=0),
}
for name, model in moons_models.items():
    result = cross_validate(model, X_moons, y_moons, cv=cv, return_train_score=True)
    print('%-28s huấn luyện %.4f, kiểm tra %.4f ± %.4f'
          % (name, result['train_score'].mean(), result['test_score'].mean(), result['test_score'].std()))

Cây quyết định               huấn luyện 1.0000, kiểm tra 0.8520 ± 0.0348
Hồi quy Logistic             huấn luyện 0.8474, kiểm tra 0.8464 ± 0.0251


Random Forest (100 cây)      huấn luyện 1.0000, kiểm tra 0.8772 ± 0.0305


AdaBoost (200 cây 1 tầng)    huấn luyện 0.9269, kiểm tra 0.8852 ± 0.0261


Gradient Boosting (200 cây)  huấn luyện 0.9838, kiểm tra 0.8724 ± 0.0341
